# Q2.3 Optimizer Showdown

In [9]:
import sys
from pathlib import Path
import argparse
import numpy as np
import wandb
from sklearn.metrics import precision_recall_fscore_support

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.data_loader import load_dataset
from src.ann.neural_network import NeuralNetwork

ENTITY = 'anandhakrishnanm21-indian-institute-of-technology-madras'


In [10]:
def to_namespace(cfg):
    d = dict(cfg)
    defaults = {
        'hidden_size': [128, 64],
        'activation': 'relu',
        'weight_init': 'xavier',
        'loss': 'cross_entropy',
        'learning_rate': 1e-3,
        'weight_decay': 0.0,
        'optimizer': 'sgd',
        'epochs': 10,
        'batch_size': 64,
    }
    defaults.update(d)
    defaults['num_layers'] = len(defaults['hidden_size'])
    return argparse.Namespace(**defaults)


def evaluate_split(model, X, Y):
    logits = model.forward(X)
    loss, acc = model.evaluate(X, Y)
    y_true = np.argmax(Y, axis=1)
    y_pred = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'loss': float(loss),
        'acc': float(acc),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'y_true': y_true,
        'y_pred': y_pred,
    }


def train_one_run(cfg, project, run_name, dataset_name='mnist', extras_fn=None):
    run = wandb.init(project=project, name=run_name, config=cfg, reinit=True)
    args = to_namespace(wandb.config)

    Xtr, Ytr, Xva, Yva, Xte, Yte = load_dataset(dataset_name)
    model = NeuralNetwork(args)

    best_val_f1 = -1.0
    for epoch in range(args.epochs):
        model.train(Xtr, Ytr, epochs=1, batch_size=args.batch_size, X_val=Xva, y_val=Yva, verbose=False)
        tr = evaluate_split(model, Xtr, Ytr)
        va = evaluate_split(model, Xva, Yva)
        te = evaluate_split(model, Xte, Yte)

        payload = {
            'epoch': epoch + 1,
            'train_loss': tr['loss'], 'train_acc': tr['acc'], 'train_f1': tr['f1'],
            'val_loss': va['loss'], 'val_acc': va['acc'], 'val_f1': va['f1'],
            'test_loss': te['loss'], 'test_acc': te['acc'], 'test_f1': te['f1'],
            'test_precision': te['precision'], 'test_recall': te['recall'],
        }
        if extras_fn is not None:
            payload.update(extras_fn(model, Xva, Yva))
        wandb.log(payload)

        if va['f1'] > best_val_f1:
            best_val_f1 = va['f1']

    wandb.summary['best_val_f1'] = float(best_val_f1)
    wandb.finish()


In [11]:
PROJECT = 'q2_3_optimizers_showdown'
wandb.finish()
base_cfg = {
    'epochs': 5,
    'batch_size': 64,
    'learning_rate': 1e-2,
    'weight_decay': 0.0,
    'hidden_size': [128, 128, 128],
    'activation': 'relu',
    'weight_init': 'xavier',
    'loss': 'cross_entropy',
}
for opt in ['sgd', 'momentum', 'nag', 'rmsprop']:
    cfg = dict(base_cfg)
    cfg['optimizer'] = opt
    train_one_run(cfg, project=PROJECT, run_name=f'optimizer_{opt}', dataset_name='mnist')


epoch,▁
test_acc,▁
test_f1,▁
test_loss,▁
test_precision,▁
test_recall,▁
train_acc,▁
train_f1,▁
train_loss,▁
val_acc,▁
+2,...


epoch,▁▃▅▆█
test_acc,▁▄▆▇█
test_f1,▁▄▆▇█
test_loss,█▄▃▁▁
test_precision,▁▄▆▇█
test_recall,▁▄▆▇█
train_acc,▁▄▆▇█
train_f1,▁▄▆▇█
train_loss,█▄▃▂▁
val_acc,▁▅▆▇█
+2,...


epoch,▁▃▅▆█
test_acc,▁▅▇█▇
test_f1,▁▅▇█▇
test_loss,█▅▂▁▂
test_precision,▁▅▇█▇
test_recall,▁▅▇█▇
train_acc,▁▄▇██
train_f1,▁▄▇██
train_loss,█▅▂▁▁
val_acc,▁▄▇▇█
+2,...


epoch,▁▃▅▆█
test_acc,▁▅▇█▇
test_f1,▁▅▇█▇
test_loss,█▄▂▁▂
test_precision,▁▅▇█▇
test_recall,▁▅▇█▇
train_acc,▁▅▇██
train_f1,▁▅▇██
train_loss,█▄▂▁▁
val_acc,▁▅▇██
+2,...


epoch,▁▃▅▆█
test_acc,▁▇█▃▃
test_f1,▁▇█▃▃
test_loss,▁▂▂█▄
test_precision,▁▇█▂▄
test_recall,▁▇█▃▃
train_acc,▁▇█▃▃
train_f1,▁▇█▄▄
train_loss,▄▁▂█▃
val_acc,▁▅█▃▂
+2,...


In [13]:
api = wandb.Api()
runs = api.runs(f'{ENTITY}/q2_3_optimizers_showdown')
for r in runs[:10]:
    print(r.name, r.summary.get('test_f1'), r.summary.get('best_val_f1'))


optimizer_sgd 0.9383556587974848 0.9340024331014298
optimizer_momentum 0.9676290689060872 0.9708040516938456
optimizer_nag 0.9704786726179696 0.9702364893916006
optimizer_rmsprop 0.941593119097176 0.9548307688920096
